In [1]:
import transformers
import torch
import sagemaker
from sagemaker.huggingface import HuggingFace
from sagemaker import get_execution_role

role = get_execution_role()
session = sagemaker.Session()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
import sys
sys.version

'3.12.9 | packaged by conda-forge | (main, Feb 14 2025, 08:00:06) [GCC 13.3.0]'

In [3]:
torch.__version__

'2.6.0'

In [4]:
huggingface_estimator = HuggingFace(
    entry_point="script.py",
    source_dir="./scripts",
    py_version="py311",
    transformers_version="4.49",
    pytorch_version="2.5",
    role=role,
    instance_count=1,
    instance_type="ml.g5.2xlarge",
    output_path="s3://news-data-bucket-42/output/",
    hyperparameters={
        'epochs':3,
        'train_batch_size':4,
        'valid_batch_size':2,
        'learning_rate':1e-04,
        'max_len':512
    },
    enable_sagemaker_metrics=True
)

In [ ]:
huggingface_estimator.fit(job_name="finetuning-distilbert-news-run-11")

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: finetuning-distilbert-news-run-11


2025-11-19 21:12:44 Starting - Starting the training job
2025-11-19 21:12:44 Pending - Training job waiting for capacity...
2025-11-19 21:13:09 Pending - Preparing the instances for training...

In [2]:
"""
import pandas as pd
from transformers import AutoTokenizer, AutoModel
"""

In [3]:
"""
s3_path = "s3://news-data-bucket-42/training/newsCorpora.csv"
model_chkpt = "distilbert/distilbert-base-uncased"

df = pd.read_csv(s3_path,sep="\t", names=["TITLE","URL","PUBLISHER","CATEGORY","STORY","HOSTNAME","TIMESTAMP"])
df = df[["TITLE","CATEGORY"]]
df = df.drop_duplicates()
df["CATEGORY"] = df["CATEGORY"].map(
    {
        "b":"Business",
        "t":"Technology",
        "e":"Entertainment",
        "m":"Health"
    }
)

df = df.sample(frac=0.1,random_state=42)
df = df.reset_index(drop=True)

print(df)

df["ENCODE_CAT"] = df["CATEGORY"].map({
    "Business":0,
    "Technology":1,
    "Entertainment":2,
    "Health":3
})
"""


                                                   TITLE       CATEGORY
0      Justin Bieber Detained at LAX, Stopped for Que...  Entertainment
1      Antarctic Ice Melt Rate Has Doubled Since 2010...     Technology
2      Asian stocks subdued on Ukraine caution in hol...       Business
3      Man crowdsources $31000 on Kickstarter to make...  Entertainment
4      Watch Three New Clips and a Featurette from 'T...  Entertainment
...                                                  ...            ...
40726    Microsoft's Xbox Entertainment Studios To Close     Technology
40727                            Ciara hosts baby shower  Entertainment
40728  Kim Kardashian's wacky wedding prep: Toe lipo ...  Entertainment
40729  Amazon Vows FTC Lawsuit Over Millions In Unaut...     Technology
40730  Asus ZenFone 6 vs. Sony Xperia T2 Ultra Specs ...     Technology

[40731 rows x 2 columns]


In [5]:
"""
model = AutoModel.from_pretrained(model_chkpt)
tokenizer = AutoTokenizer.from_pretrained(model_chkpt)
"""

In [10]:
"""
feed_forward = torch.nn.Linear(768,768)
dropout = torch.nn.Dropout(0.1)

classifier = torch.nn.Linear(768,4)
""

In [16]:

"""
title = df['TITLE'][0]
df.iloc[0,:]
"""


TITLE         Justin Bieber Detained at LAX, Stopped for Que...
CATEGORY                                          Entertainment
ENCODE_CAT                                                    2
Name: 0, dtype: object

'Justin Bieber Detained at LAX, Stopped for Questioning'

In [22]:
"""
inputs = tokenizer.encode_plus(
    title,
    add_special_tokens=True,
    max_length=20,
    padding="max_length",
    truncation=True,
    return_token_type_ids=True,
    return_attention_mask=True,
    return_tensors="pt"
)
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']
"""

In [23]:
"""
output_1 = model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
hidden_state = output_1[0]
print(hidden_state)
pooler = hidden_state[:,0]
print(pooler)
pooler = feed_forward(pooler)
pooler = torch.nn.ReLU()(pooler)
pooler = dropout(pooler)
output = classifier(pooler)
print(output)
"""


tensor([[[-0.0361, -0.2070, -0.2416,  ...,  0.0268,  0.2496,  0.0363],
         [ 0.5981, -0.0290,  0.1936,  ...,  0.2186,  0.3155, -0.7290],
         [ 0.8356, -1.0539,  0.8764,  ..., -0.1108,  0.3754,  0.3937],
         ...,
         [ 0.1606, -0.1122,  0.1525,  ...,  0.0951, -0.1427,  0.0923],
         [ 0.2845, -0.0313,  0.2001,  ...,  0.0661, -0.1239,  0.0117],
         [ 0.1610, -0.1357,  0.0525,  ...,  0.0566, -0.1489, -0.0093]]],
       grad_fn=<NativeLayerNormBackward0>)
tensor([[-3.6077e-02, -2.0696e-01, -2.4159e-01,  4.3672e-02,  3.1646e-01,
         -7.0331e-02,  3.2082e-01,  3.4403e-01, -9.5487e-02,  3.0510e-03,
          8.3152e-02, -3.3916e-01, -4.3141e-01,  2.2889e-01,  1.2434e-01,
          1.2346e-01, -4.1715e-01,  8.5984e-02,  2.8407e-01, -5.7542e-02,
          1.0905e-01, -3.7198e-01,  1.6863e-01, -1.3862e-01,  6.1072e-02,
         -2.1776e-01, -2.6934e-01, -1.3906e-02, -3.2368e-01,  9.5729e-02,
          1.6644e-01,  3.1923e-01, -3.6952e-01, -5.1654e-02,  2.9580e-0

In [24]:
"""
pooler.shape,hidden_state.shape,output.shape
"""

torch.Size([1, 20, 768])

torch.Size([1, 768])

torch.Size([1, 4])